# Food Delivery Data & AI Pipeline
## Deliverable 1 — Kafka Ingestion and Quarantine

This notebook adapts the Day 2 **Real Kafka Round Trip** lab to a food-delivery use case.

A real Kafka producer sends order events to `food_orders_raw`.  
A real Kafka consumer validates every event with a Pydantic data contract.

- Valid records are routed to `food_orders_valid`.
- Invalid records are routed to `food_orders_quarantine`.
- Every quarantined record includes the original payload and the rejection reason.

### Pipeline

```text
KafkaProducer
     ↓
food_orders_raw
     ↓
Pydantic schema gate
   /                \
valid              invalid
  ↓                   ↓
food_orders_valid   food_orders_quarantine
```

The notebook later reads both output topics to prove that the records were actually routed through Kafka.

## 1. Environment Setup

Google Colab does not include a running Kafka broker by default.

The following cells install the same Python libraries used in the Day 2 lab and start one local Kafka broker at `localhost:9092`.

In [25]:
!pip install kafka-python pydantic

In [26]:
# Download Kafka 3.7.0 from the official Apache archive.
!curl -sSLO https://archive.apache.org/dist/kafka/3.7.0/kafka_2.13-3.7.0.tgz
!tar -xzf kafka_2.13-3.7.0.tgz

# Format Kafka's KRaft storage.
!cd kafka_2.13-3.7.0 && \
  bin/kafka-storage.sh format \
  -t $(bin/kafka-storage.sh random-uuid) \
  -c config/kraft/server.properties \
  --ignore-formatted

# Start the Kafka broker in the background.
!cd kafka_2.13-3.7.0 && \
  nohup bin/kafka-server-start.sh config/kraft/server.properties \
  > /tmp/kafka.log 2>&1 &

import time

print("Waiting for Kafka to start...")
time.sleep(20)
print("Kafka startup wait completed.")

Exception in thread "main" java.lang.RuntimeException: Invalid cluster.id in: /tmp/kraft-combined-logs/meta.properties. Expected 5CivqM8uTr-yWbDnmzJ5Ig, but read v8lw_tZ4RnyBuoiCtUP9LA
	at org.apache.kafka.metadata.properties.MetaPropertiesEnsemble.verify(MetaPropertiesEnsemble.java:516)
	at kafka.tools.StorageTool$.formatCommand(StorageTool.scala:433)
	at kafka.tools.StorageTool$.main(StorageTool.scala:95)
	at kafka.tools.StorageTool.main(StorageTool.scala)
Waiting for Kafka to start...
Kafka startup wait completed.


## 2. Imports, Topics, and Data Contract

The imports and producer/consumer style intentionally stay close to the Day 2 lab.

The main addition is a quarantine topic, which is required by the capstone rubric.

In [27]:
import json
import time
from datetime import datetime, timezone

from pydantic import BaseModel, ValidationError, field_validator

BOOTSTRAP_SERVERS = "localhost:9092"

RAW_TOPIC = "food_orders_raw"
VALID_TOPIC = "food_orders_valid"
QUARANTINE_TOPIC = "food_orders_quarantine"

In [28]:
class OrderEvent(BaseModel):
    """
    Data contract for one food-delivery order event.

    Every incoming Kafka message must contain these fields with the
    expected data types and valid business values.
    """

    order_id: str
    customer_id: str
    restaurant_name: str
    customer_area: str
    order_total_sar: float
    delivery_distance_km: float
    estimated_delivery_minutes: int
    status: str
    event_ts: str

    @field_validator(
        "order_id",
        "customer_id",
        "restaurant_name",
        "customer_area",
    )
    @classmethod
    def text_must_not_be_empty(cls, value: str) -> str:
        if not value.strip():
            raise ValueError("field must not be empty")
        return value.strip()

    @field_validator("order_total_sar")
    @classmethod
    def valid_order_total(cls, value: float) -> float:
        if not (0 < value <= 5000):
            raise ValueError(
                "order_total_sar must be greater than 0 and at most 5000"
            )
        return value

    @field_validator("delivery_distance_km")
    @classmethod
    def valid_delivery_distance(cls, value: float) -> float:
        if not (0 < value <= 50):
            raise ValueError(
                "delivery_distance_km must be greater than 0 and at most 50"
            )
        return value

    @field_validator("estimated_delivery_minutes")
    @classmethod
    def valid_delivery_time(cls, value: int) -> int:
        if not (5 <= value <= 180):
            raise ValueError(
                "estimated_delivery_minutes must be between 5 and 180"
            )
        return value

    @field_validator("status")
    @classmethod
    def valid_status(cls, value: str) -> str:
        allowed_statuses = {
            "CONFIRMED",
            "PREPARING",
            "PICKED_UP",
            "DELIVERED",
            "CANCELLED",
        }

        normalized = value.strip().upper()

        if normalized not in allowed_statuses:
            raise ValueError(
                f"status must be one of {sorted(allowed_statuses)}"
            )

        return normalized

    @field_validator("event_ts")
    @classmethod
    def valid_timestamp(cls, value: str) -> str:
        try:
            datetime.fromisoformat(value.replace("Z", "+00:00"))
        except ValueError as error:
            raise ValueError(
                "event_ts must use a valid ISO-8601 format"
            ) from error

        return value

## 3. Producer

The producer sends:

- Four valid food-order records
- Four deliberately malformed records

The bad records prove different failure paths:

- Negative order total
- Unsupported status
- Blank restaurant name
- Wrong data type for order total

In [29]:
def produce_sample_orders():
    from kafka import KafkaProducer

    producer = KafkaProducer(
        bootstrap_servers=BOOTSTRAP_SERVERS,
        value_serializer=lambda value: json.dumps(value).encode("utf-8"),
    )

    good = [
        {
            "order_id": "ORD-1001",
            "customer_id": "CUS-501",
            "restaurant_name": "Najd Kitchen",
            "customer_area": "Al Olaya",
            "order_total_sar": 86.50,
            "delivery_distance_km": 6.2,
            "estimated_delivery_minutes": 34,
            "status": "CONFIRMED",
            "event_ts": "2026-07-22T17:10:00Z",
        },
        {
            "order_id": "ORD-1002",
            "customer_id": "CUS-502",
            "restaurant_name": "Olive and Thyme",
            "customer_area": "Al Yasmin",
            "order_total_sar": 54.00,
            "delivery_distance_km": 4.8,
            "estimated_delivery_minutes": 26,
            "status": "PREPARING",
            "event_ts": "2026-07-22T17:10:01Z",
        },
        {
            "order_id": "ORD-1003",
            "customer_id": "CUS-503",
            "restaurant_name": "Urban Grill",
            "customer_area": "Al Nakheel",
            "order_total_sar": 112.75,
            "delivery_distance_km": 8.4,
            "estimated_delivery_minutes": 41,
            "status": "PICKED_UP",
            "event_ts": "2026-07-22T17:10:02Z",
        },
        {
            "order_id": "ORD-1004",
            "customer_id": "CUS-504",
            "restaurant_name": "Palm Bakery",
            "customer_area": "Al Malaz",
            "order_total_sar": 39.25,
            "delivery_distance_km": 3.1,
            "estimated_delivery_minutes": 22,
            "status": "DELIVERED",
            "event_ts": "2026-07-22T17:10:03Z",
        },
    ]

    bad = [
        {
            "order_id": "ORD-1005",
            "customer_id": "CUS-505",
            "restaurant_name": "Desert Bowl",
            "customer_area": "Al Murabba",
            "order_total_sar": -25.00,
            "delivery_distance_km": 5.5,
            "estimated_delivery_minutes": 30,
            "status": "CONFIRMED",
            "event_ts": "2026-07-22T17:10:04Z",
        },
        {
            "order_id": "ORD-1006",
            "customer_id": "CUS-506",
            "restaurant_name": "Garden Table",
            "customer_area": "Al Wurud",
            "order_total_sar": 73.25,
            "delivery_distance_km": 7.0,
            "estimated_delivery_minutes": 38,
            "status": "ON_THE_MOON",
            "event_ts": "2026-07-22T17:10:05Z",
        },
        {
            "order_id": "ORD-1007",
            "customer_id": "CUS-507",
            "restaurant_name": "   ",
            "customer_area": "Al Sulimaniyah",
            "order_total_sar": 49.50,
            "delivery_distance_km": 2.8,
            "estimated_delivery_minutes": 19,
            "status": "PREPARING",
            "event_ts": "2026-07-22T17:10:06Z",
        },
        {
            "order_id": "ORD-1008",
            "customer_id": "CUS-508",
            "restaurant_name": "Riyadh Roastery",
            "customer_area": "King Fahd District",
            "order_total_sar": "free",
            "delivery_distance_km": 6.6,
            "estimated_delivery_minutes": 35,
            "status": "CONFIRMED",
            "event_ts": "2026-07-22T17:10:07Z",
        },
    ]

    for message in good + bad:
        metadata = producer.send(RAW_TOPIC, message).get(timeout=10)

        print(
            f"[PRODUCER] sent {message.get('order_id')} "
            f"to {metadata.topic} @ offset {metadata.offset}"
        )

    producer.flush()
    producer.close()

    return len(good) + len(bad)

## 4. Consumer, Validation, and Quarantine

The consumer is the ingestion boundary.

For every raw Kafka message:

1. `OrderEvent.model_validate()` checks the data contract.
2. A valid record is sent to `food_orders_valid`.
3. An invalid record is sent to `food_orders_quarantine`.
4. The quarantine event stores the original record and exact rejection reason.

In [30]:
def consume_validate_and_route(max_messages: int = 8, timeout_s: int = 15):
    from kafka import KafkaConsumer, KafkaProducer

    consumer = KafkaConsumer(
        RAW_TOPIC,
        bootstrap_servers=BOOTSTRAP_SERVERS,
        auto_offset_reset="earliest",
        enable_auto_commit=True,
        group_id=f"food-order-schema-check-{int(time.time())}",
        value_deserializer=lambda value: json.loads(
            value.decode("utf-8")
        ),
        consumer_timeout_ms=timeout_s * 1000,
    )

    router = KafkaProducer(
        bootstrap_servers=BOOTSTRAP_SERVERS,
        value_serializer=lambda value: json.dumps(value).encode("utf-8"),
    )

    accepted = 0
    quarantined = 0

    for record in consumer:
        try:
            validated_order = OrderEvent.model_validate(record.value)

            router.send(
                VALID_TOPIC,
                validated_order.model_dump(),
            )

            accepted += 1

            print(
                f"[VALID] accepted @offset {record.offset}: "
                f"{validated_order.order_id}"
            )

        except ValidationError as error:
            rejection_reason = "; ".join(
                f"{'.'.join(map(str, item['loc']))}: {item['msg']}"
                for item in error.errors()
            )

            quarantine_message = {
                "original_record": record.value,
                "rejection_reason": rejection_reason,
                "source_topic": record.topic,
                "source_partition": record.partition,
                "source_offset": record.offset,
                "rejected_at": datetime.now(
                    timezone.utc
                ).isoformat(),
            }

            router.send(
                QUARANTINE_TOPIC,
                quarantine_message,
            )

            quarantined += 1

            print(
                f"[QUARANTINED] rejected @offset {record.offset}: "
                f"{record.value.get('order_id')} | "
                f"{rejection_reason}"
            )

        if accepted + quarantined >= max_messages:
            break

    router.flush()
    router.close()
    consumer.close()

    print(
        f"\nSummary: {accepted} accepted, "
        f"{quarantined} quarantined."
    )

    return accepted, quarantined

## 5. Verification

The following helper reads the two destination topics.

This proves that valid and invalid records were routed through the real Kafka broker instead of only being classified in Python memory.

In [31]:
def read_topic(topic: str, expected_messages: int, timeout_s: int = 15):
    from kafka import KafkaConsumer

    consumer = KafkaConsumer(
        topic,
        bootstrap_servers=BOOTSTRAP_SERVERS,
        auto_offset_reset="earliest",
        enable_auto_commit=False,
        group_id=f"verify-{topic}-{int(time.time())}",
        value_deserializer=lambda value: json.loads(
            value.decode("utf-8")
        ),
        consumer_timeout_ms=timeout_s * 1000,
    )

    messages = []

    for record in consumer:
        messages.append(record.value)

        if len(messages) >= expected_messages:
            break

    consumer.close()
    return messages

In [32]:
def main():
    print(
        "Food delivery Kafka ingestion "
        f"(raw topic: {RAW_TOPIC})"
    )

    try:
        produced = produce_sample_orders()
        time.sleep(1)

        accepted, quarantined = consume_validate_and_route(
            max_messages=produced
        )

        time.sleep(1)

        valid_records = read_topic(
            VALID_TOPIC,
            accepted,
        )

        quarantine_records = read_topic(
            QUARANTINE_TOPIC,
            quarantined,
        )

        print("\n--- VALID TOPIC ---")
        for record in valid_records:
            print(
                record["order_id"],
                "|",
                record["restaurant_name"],
                "|",
                record["status"],
            )

        print("\n--- QUARANTINE TOPIC ---")
        for record in quarantine_records:
            print(
                record["original_record"].get("order_id"),
                "|",
                record["rejection_reason"],
            )

        assert produced == 8
        assert accepted == 4
        assert quarantined == 4
        assert len(valid_records) == accepted
        assert len(quarantine_records) == quarantined
        assert all(
            record["rejection_reason"]
            for record in quarantine_records
        )

        print(
            "\nDELIVERABLE 1 PASSED: "
            "4 valid records and 4 malformed records "
            "were routed correctly."
        )

    except ImportError as error:
        print(f"\nMissing dependency: {error}")
        print(
            "Install it first with: "
            "pip install kafka-python pydantic"
        )

    except Exception as error:
        print(
            f"\nCould not complete the Kafka round trip "
            f"at {BOOTSTRAP_SERVERS}: {error}"
        )
        print(
            "Confirm that the local Kafka broker is running, "
            "then rerun the notebook."
        )

main()

Food delivery Kafka ingestion (raw topic: food_orders_raw)
[PRODUCER] sent ORD-1001 to food_orders_raw @ offset 8
[PRODUCER] sent ORD-1002 to food_orders_raw @ offset 9
[PRODUCER] sent ORD-1003 to food_orders_raw @ offset 10
[PRODUCER] sent ORD-1004 to food_orders_raw @ offset 11
[PRODUCER] sent ORD-1005 to food_orders_raw @ offset 12
[PRODUCER] sent ORD-1006 to food_orders_raw @ offset 13
[PRODUCER] sent ORD-1007 to food_orders_raw @ offset 14
[PRODUCER] sent ORD-1008 to food_orders_raw @ offset 15


/tmp/ipykernel_686/4028916504.py:8: DeprecationWarning: value_serializer does not implement kafka.serializer.Serializer
  produced = produce_sample_orders()
/tmp/ipykernel_686/3463557843.py:4: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(
/tmp/ipykernel_686/4028916504.py:11: DeprecationWarning: value_serializer does not implement kafka.serializer.Serializer
  accepted, quarantined = consume_validate_and_route(


[VALID] accepted @offset 0: ORD-1001
[VALID] accepted @offset 1: ORD-1002
[VALID] accepted @offset 2: ORD-1003
[VALID] accepted @offset 3: ORD-1004
[QUARANTINED] rejected @offset 4: ORD-1005 | order_total_sar: Value error, order_total_sar must be greater than 0 and at most 5000
[QUARANTINED] rejected @offset 5: ORD-1006 | status: Value error, status must be one of ['CANCELLED', 'CONFIRMED', 'DELIVERED', 'PICKED_UP', 'PREPARING']
[QUARANTINED] rejected @offset 6: ORD-1007 | restaurant_name: Value error, field must not be empty
[QUARANTINED] rejected @offset 7: ORD-1008 | order_total_sar: Input should be a valid number, unable to parse string as a number

Summary: 4 accepted, 4 quarantined.

--- VALID TOPIC ---
ORD-1001 | Najd Kitchen | CONFIRMED
ORD-1002 | Olive and Thyme | PREPARING
ORD-1003 | Urban Grill | PICKED_UP
ORD-1004 | Palm Bakery | DELIVERED

--- QUARANTINE TOPIC ---
ORD-1005 | order_total_sar: Value error, order_total_sar must be greater than 0 and at most 5000
ORD-1006 | st

# Technical Documentation

## Technologies

- Apache Kafka 3.7.0
- `kafka-python`
- Pydantic v2
- Google Colab
- Python

## Kafka topics

| Topic | Purpose |
|---|---|
| `food_orders_raw` | Stores producer events exactly as submitted |
| `food_orders_valid` | Stores records that passed the Pydantic contract |
| `food_orders_quarantine` | Stores malformed records with rejection evidence |

## Failure-path evidence

The notebook intentionally sends four malformed events. Each invalid event is:

1. Rejected by Pydantic
2. Published to the quarantine topic
3. Stored with the original payload
4. Stored with the rejection reason
5. Read back from Kafka during verification

## Expected output

```text
8 messages produced
4 records accepted
4 records quarantined
```
